# Lab 7 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/viz.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Work it out

**Not submitted, and not written at the keyboard.** Work these in pairs, out loud, one of you holding the rubric card. The rubric cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/viz.html>.

### A1. Which number goes on the slide

*Suggested: 5 minutes.*

A neighborhood has 100 listings. Ninety rent for about \$2,000; the other ten
are penthouses at \$9,000.

a. Roughly what is the mean? The median? Both are doable in your head.
b. The slide says **"average rent: \$2,700"**. How many of the 100 listings rent
   for anything near that?
c. Which number would you put on the slide, and what would you put beside it so
   a reader is not misled?

### A2. Two bars, and what is under them

*Suggested: 6 minutes.*

A slide compares two neighborhoods with a bar chart of mean rent.

| | Shaw | Petworth |
|:--|--:|--:|
| mean rent | \$2,180 | \$1,760 |
| listings | 140 | 12 |
| standard deviation | \$520 | \$480 |

a. The bars differ by \$420. Is that a large difference compared with how much
   rent varies *within* each neighborhood?
b. A renter asks, "will I pay more in Shaw?" What is the honest answer from
   these three rows?
c. Name two things the bar chart does not show that you would add.

### A3. The pattern that reverses

*Suggested: 6 minutes.*

Across a whole city, rent and square footage correlate at $r = 0.72$. Inside
each neighborhood, the correlation is about $0.15$.

a. What can produce that, with nothing whatsoever wrong in the data?
b. You are advising a Petworth landlord on what an extra 100 square feet is
   worth. Which number do you use, and why?
c. What single figure would show the reversal?

### A4. What would have caught it

*Suggested: 5 minutes.*

A listings file arrives with `rent`, `sqft`, `beds` and `built`. Four rows are
wrong: a rent of \$0, a `sqft` of 12, a `built` year of 2103, and a listing with
9 beds in 400 square feet.

a. For each, write the one-line check that catches it.
b. Which of the four would you catch if you only ever looked at the **mean** of
   each column? Why?
c. Where in your workflow do these checks belong, and what should they do when
   one fires?

## Part B—Build

### B1. Shape before summary

*Suggested: 8 minutes.*

`listings` holds 400 apartments in three neighborhoods. Before any average, ask
what the column looks like.

- `center`---a tuple `(mean, median)` of `rent`;
- `gap`---the mean minus the median;
- `within_250`---the share of listings whose rent is within \$250 of the mean;
- `tail`---how many listings rent for more than \$4,500.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.precision", 2)
rng = np.random.default_rng(6103)
n = 400
hood = rng.choice(["Shaw", "Petworth", "Georgetown"], n, p=[0.45, 0.35, 0.20])
base = {"Shaw": 2100, "Petworth": 1700, "Georgetown": 3400}
typical = {"Shaw": 750, "Petworth": 600, "Georgetown": 1100}
sqft = (np.array([typical[h] for h in hood]) + rng.normal(0, 140, n)).clip(300, 1800)
rent = (np.array([base[h] for h in hood])
        + 0.5 * (sqft - np.array([typical[h] for h in hood])) + rng.normal(0, 350, n))
premium = rng.random(n) < 0.05
rent = np.where(premium, rent + rng.uniform(3000, 5000, n), rent)
listings = pd.DataFrame({"rent": np.round(rent / 25) * 25, "sqft": np.round(sqft),
                         "hood": hood, "beds": rng.integers(0, 4, n)})

In [ ]:
center = ...
gap = ...
within_250 = ...
tail = ...

print(f"mean {center[0]:,.0f}   median {center[1]:,.0f}   gap {gap:,.0f}")
print(f"within $250 of the mean: {within_250:.1%}    above $4,500: {tail}")

In [ ]:
# self-check: run this after your answer above
wm = float(listings["rent"].mean())
wmed = float(listings["rent"].median())
assert len(center) == 2, "center should be a (mean, median) pair"
assert abs(float(center[0]) - wm) < 1e-6 and abs(float(center[1]) - wmed) < 1e-6, \
    "center is not the mean and median of rent"
assert abs(float(gap) - (wm - wmed)) < 1e-6, "gap should be the mean minus the median"
w = float(((listings["rent"] - wm).abs() <= 250).mean())
assert abs(float(within_250) - w) < 1e-9, \
    "within_250 should be the share of listings within $250 of the mean"
assert int(tail) == int((listings["rent"] > 4500).sum()), \
    "tail should count the listings renting for more than $4,500"
print("looks right")

### B2. The bin width writes the story

*Suggested: 10 minutes.*

Draw `rent` three times---8 bins, 30 bins, 120 bins---side by side, then record
what survived.

- `clumps`---a dict with keys `8`, `30` and `120`, each holding the number of
  separated clumps **you** see at that width. It is a judgment call, and the
  check only asks that you looked at all three;
- `widest_gap`---the largest dollar jump between two consecutive sorted rents;
- `gap_at`---the rent that jump starts from.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
for a, b in zip(ax, [8, 30, 120]):
    a.hist(listings["rent"], bins=b, color="#7b2d26")
    a.set_title(f"{b} bins")
    a.set_xlabel("rent")
plt.tight_layout()
plt.show()

clumps = ...
widest_gap = ...
gap_at = ...

print(clumps, widest_gap, gap_at)

In [ ]:
# self-check: run this after your answer above
assert isinstance(clumps, dict) and set(clumps) == {8, 30, 120}, \
    "clumps should be a dict with keys 8, 30 and 120"
assert all(isinstance(v, (int, np.integer)) and v >= 1 for v in clumps.values()), \
    "each entry should be how many separated clumps you counted, as an integer"
s = np.sort(listings["rent"].to_numpy())
d = np.diff(s)
assert abs(float(widest_gap) - float(d.max())) < 1e-6, \
    "widest_gap should be the largest jump between consecutive sorted rents"
assert abs(float(gap_at) - float(s[int(d.argmax())])) < 1e-6, \
    "gap_at should be the rent the widest jump starts from"
print("looks right")

### B3. Count, center, spread

*Suggested: 10 minutes.*

Summarize rent by neighborhood the way A2 said a bar chart should have.

- `table`---a DataFrame indexed by `hood` with columns `n`, `median`, `iqr`, in
  that order, sorted by `median` from high to low;
- `overlap`---the share of Petworth listings that cost more than the **median**
  Shaw listing.

In [ ]:
table = ...
overlap = ...

print(table)
print(f"Petworth listings above Shaw's median: {overlap:.1%}")

In [ ]:
# self-check: run this after your answer above
g = listings.groupby("hood")["rent"]
want = pd.DataFrame({"n": g.size(), "median": g.median(),
                     "iqr": g.quantile(0.75) - g.quantile(0.25)})
want = want.sort_values("median", ascending=False)
assert isinstance(table, pd.DataFrame), \
    f"table should be a DataFrame, got {type(table).__name__}"
assert list(table.columns) == ["n", "median", "iqr"], \
    "table should have columns n, median, iqr, in that order"
assert list(table.index) == list(want.index), \
    "table should be sorted by median, highest first"
assert np.allclose(table.to_numpy().astype(float), want.to_numpy().astype(float)), \
    "the numbers do not match"
shaw_med = float(listings.loc[listings["hood"] == "Shaw", "rent"].median())
w = float((listings.loc[listings["hood"] == "Petworth", "rent"] > shaw_med).mean())
assert abs(float(overlap) - w) < 1e-9, \
    "overlap should be the share of Petworth listings above Shaw's median rent"
print("looks right")

### B4. The correlation that belongs to the neighborhoods

*Suggested: 12 minutes.*

A3 on the real thing. Rent against square footage, pooled and within groups.

- `overall`---the correlation between `rent` and `sqft` across all 400 listings;
- `by_hood`---a Series of that correlation inside each neighborhood, indexed by
  `hood`;
- `spread_of_bases`---the standard deviation of the three neighborhood **median**
  rents, which is the between-group variation the pooled number picks up.

Then draw the scatter, colored by neighborhood, and see whether it looks like
what you predicted in A3.

In [ ]:
overall = ...
by_hood = ...
spread_of_bases = ...

fig, ax = plt.subplots(figsize=(5, 3.5))
for h, c in zip(["Shaw", "Petworth", "Georgetown"], ["#7b2d26", "#2d5d7b", "#4a7b2d"]):
    part = listings[listings["hood"] == h]
    ax.scatter(part["sqft"], part["rent"], s=12, alpha=0.7, label=h, color=c)
ax.set_xlabel("square feet")
ax.set_ylabel("monthly rent ($)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"pooled {overall:.3f}")
print(by_hood.round(3))
print(f"spread of the three medians: {spread_of_bases:,.0f}")

In [ ]:
# self-check: run this after your answer above
wo = float(listings["rent"].corr(listings["sqft"]))
wb = listings.groupby("hood")[["rent", "sqft"]].apply(
    lambda d: d["rent"].corr(d["sqft"]))
assert abs(float(overall) - wo) < 1e-6, \
    "overall should be the correlation across all 400 listings"
assert isinstance(by_hood, pd.Series), \
    f"by_hood should be a Series, got {type(by_hood).__name__}"
assert set(by_hood.index) == set(wb.index), "by_hood should be indexed by hood"
assert np.allclose(by_hood.sort_index().to_numpy(), wb.sort_index().to_numpy()), \
    "by_hood does not match the within-neighborhood correlations"
ws = float(listings.groupby("hood")["rent"].median().std())
assert abs(float(spread_of_bases) - ws) < 1e-6, \
    "spread_of_bases should be the standard deviation of the three median rents"
assert float(overall) > float(by_hood.max()), \
    "the pooled correlation should exceed every within-group one; check how you grouped"
print("looks right")

### B5. Four rows no average will show you

*Suggested: 10 minutes.*

`dirty` is the same 400 listings with a `built` year added and the four bad
rows from A4 appended.

- `flags`---a dict with keys `"rent"`, `"sqft"`, `"built"` and `"crowded"`, each
  holding the **number of rows** failing that check: non-positive rent, `sqft`
  under 100, `built` after 2026, and more than one bed per 80 square feet;
- `worst`---the `built` year furthest in the future;
- `clean`---`dirty` with every flagged row removed.

In [ ]:
bad = pd.DataFrame({"rent": [0.0, 2400.0, 1950.0, 2100.0],
                    "sqft": [900.0, 12.0, 800.0, 400.0],
                    "hood": ["Shaw", "Petworth", "Shaw", "Georgetown"],
                    "beds": [2, 1, 1, 9],
                    "built": [1990, 2001, 2103, 1975]})
dirty = pd.concat([listings.assign(built=rng.integers(1890, 2025, n)), bad],
                  ignore_index=True)

In [ ]:
flags = ...
worst = ...
clean = ...

print(flags, "| worst built year:", worst)
print(len(dirty), "rows ->", len(clean))
print(f"mean rent {dirty['rent'].mean():,.2f} -> {clean['rent'].mean():,.2f}")

In [ ]:
# self-check: run this after your answer above
bad_rent = dirty["rent"] <= 0
bad_sqft = dirty["sqft"] < 100
bad_built = dirty["built"] > 2026
crowded = dirty["beds"] / dirty["sqft"] > 1 / 80
assert isinstance(flags, dict) and set(flags) == {"rent", "sqft", "built", "crowded"}, \
    "flags should have keys rent, sqft, built and crowded"
for key, mask in [("rent", bad_rent), ("sqft", bad_sqft),
                  ("built", bad_built), ("crowded", crowded)]:
    assert int(flags[key]) == int(mask.sum()), \
        f"flags['{key}'] should be {int(mask.sum())}, not {flags[key]}"
assert int(worst) == int(dirty["built"].max()), \
    "worst should be the built year furthest in the future"
keep = ~(bad_rent | bad_sqft | bad_built | crowded)
assert len(clean) == int(keep.sum()), f"clean should have {int(keep.sum())} rows"
assert float(clean["rent"].min()) > 0 and float(clean["sqft"].min()) >= 100, \
    "clean still contains a flagged row"
print("looks right")

### B6. One figure, defensible

*Suggested: 12 minutes.*

Draw the figure you would put in front of a renter choosing between Shaw and
Petworth, and write its caption. `two` holds just those two neighborhoods.

Requirements: show the **distributions**, not two bars; label both axes; and let
the caption carry the numbers a reader needs.

- `caption`---a string giving each neighborhood's `n` and median, and the
  overlap from B3;
- `fig_ok`---`True` once your figure shows both neighborhoods' spreads.

In [ ]:
two = listings[listings["hood"].isin(["Shaw", "Petworth"])]

In [ ]:
# A box plot, a violin, or the points with jitter all qualify. A bar does not.
fig, ax = plt.subplots(figsize=(5, 3.5))
...
plt.tight_layout()
plt.show()

caption = ...
fig_ok = ...

print(caption)

In [ ]:
# self-check: run this after your answer above
import re
shaw = two.loc[two["hood"] == "Shaw", "rent"]
pet = two.loc[two["hood"] == "Petworth", "rent"]
assert isinstance(caption, str) and len(caption) > 40, \
    "caption should be a sentence or two, as a string"
assert "Shaw" in caption and "Petworth" in caption, "name both neighborhoods"
nums = [int(x.replace(",", "")) for x in re.findall(r"\d[\d,]*", caption)]
assert len(shaw) in nums, f"the caption should state Shaw's n ({len(shaw)})"
assert len(pet) in nums, f"the caption should state Petworth's n ({len(pet)})"
for med in [shaw.median(), pet.median()]:
    assert any(abs(v - med) <= 25 for v in nums), \
        f"the caption should state the median {med:,.0f}"
share = 100 * (pet > shaw.median()).mean()
assert any(abs(v - share) <= 1.5 for v in nums), \
    "the caption should state the overlap from B3, as a percentage"
assert bool(fig_ok) is True, "set fig_ok to True once the figure shows both spreads"
print("looks right")

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/viz.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.